# 06 — Subqueries & CTEs

**What's in this notebook:**
- Subquery basics — what a subquery can return (scalar, row, table) and where it can appear
- Scalar subqueries in `SELECT`, and table subqueries in `FROM` (derived tables)
- Subqueries in `WHERE` — `IN`, `NOT IN`, `EXISTS`, `NOT EXISTS`, and how they differ on `NULL`
- Correlated vs uncorrelated — when the inner query references the outer query's columns
- CTEs with `WITH` — turning nested queries into readable, step-by-step pipelines
- Recursive CTEs — traversing hierarchies and graphs with `WITH RECURSIVE`
- Common pitfalls to carry forward

This notebook uses `customers`, `orders`, `prospects`, and `employees` from earlier notebooks.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. Subquery basics — what they return, where they go

A **subquery** is a `SELECT` statement nested inside another statement. The outer query treats the subquery's result as a value, a row, or a table — depending on how it is used.

Three shapes of subquery, by what they return:

- **Scalar subquery** — returns exactly one row, one column. Usable anywhere a value is allowed: in `SELECT`, in `WHERE`, in an expression.
- **Row subquery** — returns one row, multiple columns. Used with row-comparison constructs like `(a, b) = (SELECT x, y FROM ...)`.
- **Table subquery** — returns many rows, many columns. Used in `FROM` (as a derived table) or in `WHERE ... IN (...)`.

Three shapes of subquery, by where they go:

- **In `SELECT`** — almost always scalar, computed once per outer row (or once total, if uncorrelated).
- **In `FROM`** — table subquery, treated as a virtual table. Must have an alias.
- **In `WHERE`** — predicates like `IN`, `EXISTS`, or scalar comparisons.

Whether the subquery is **correlated** (references columns from the outer query) is a separate axis — covered in §4.

## 2. Scalar subqueries and derived tables

**Scalar subqueries** are the simplest case — they return a single value that drops into any expression context. The engine enforces the one-row-one-column constraint at runtime; returning more rows is an error.

**Derived tables** — subqueries in `FROM` — let you treat the result of one query as the input rows of another. They are how you build pipelines before CTEs were widely supported. Every derived table must have an **alias**; without one, the engine has no name to bind its columns to.

In [ ]:
%%sql
-- Scalar subquery in SELECT: each order alongside the all-orders average,
-- and how this row deviates from it.
SELECT order_id,
       amount,
       (SELECT ROUND(AVG(amount), 2) FROM orders) AS overall_avg,
       amount - (SELECT AVG(amount) FROM orders)  AS diff_from_avg
FROM orders
ORDER BY order_id;

In [ ]:
%%sql
-- Derived table in FROM: pre-aggregate, then join onto customers.
-- The alias `t` is required.
SELECT c.name, t.order_count, t.total
FROM customers c
JOIN (
    SELECT customer_id,
           COUNT(*)    AS order_count,
           SUM(amount) AS total
    FROM orders
    GROUP BY customer_id
) t ON t.customer_id = c.customer_id
ORDER BY t.total DESC;

## 3. Subqueries in WHERE — IN, EXISTS, and `NULL`

Four common predicates use a subquery on the right-hand side:

- **`x IN (subquery)`** — true if `x` equals any value the subquery returns.
- **`x NOT IN (subquery)`** — true if `x` differs from every value the subquery returns. **Inherits the `NULL` trap from notebook 03** — any `NULL` in the subquery's result makes the whole predicate `UNKNOWN`, so no rows come back.
- **`EXISTS (subquery)`** — true if the subquery returns at least one row. The actual column values don't matter — `SELECT 1` works fine.
- **`NOT EXISTS (subquery)`** — true if the subquery returns zero rows. **Handles `NULL` correctly**, which is why most experienced SQL writers prefer `NOT EXISTS` over `NOT IN`.

Rule of thumb: use `EXISTS` / `NOT EXISTS` when you are asking "are there any matching rows?". Use `IN` when you are asking "is this value in a small static list?".

In [ ]:
%%sql
-- Customers who placed at least one order, via IN.
SELECT name
FROM customers
WHERE customer_id IN (SELECT customer_id FROM orders);

In [ ]:
%%sql
-- Same question, EXISTS form. The SELECT list inside is irrelevant — only
-- the presence of a matching row matters.
SELECT name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
);

In [ ]:
%%sql
-- Customers with NO orders — NOT EXISTS handles NULL safely.
-- This is the right replacement for NOT IN when the subquery column is nullable.
SELECT name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
);

## 4. Correlated vs uncorrelated

A subquery is **uncorrelated** if it can be evaluated on its own, with no reference to the outer query. It runs **once**, and the outer query reuses its result for every row.

A subquery is **correlated** if it references one or more columns from the outer query. Conceptually it runs **once per outer row**, with the outer row's values substituted in each time. In practice, a good planner will often rewrite the correlated form into a join, but you should reason about it as if it ran per-row, because that's the mental model that explains the output.

The classic correlated pattern is "rows that compare to a per-group statistic" — orders that are bigger than the average for *their* customer, products priced above the average for *their* category, employees paid above the average for *their* department.

In [ ]:
%%sql
-- Correlated subquery: orders larger than the average order size
-- for that order's own customer.
SELECT o.order_id,
       o.customer_id,
       o.amount,
       (SELECT ROUND(AVG(amount), 2)
        FROM orders o2
        WHERE o2.customer_id = o.customer_id) AS customer_avg
FROM orders o
WHERE o.amount > (
    SELECT AVG(amount) FROM orders o2 WHERE o2.customer_id = o.customer_id
)
ORDER BY o.customer_id, o.order_id;

## 5. CTEs — naming the steps of a pipeline

A **Common Table Expression** (CTE) is a named subquery declared at the top of a statement with `WITH`. It serves the same role as a derived table in `FROM`, but with three advantages:

- **A name** — you can refer to it by that name as many times as you like in the main query (and, with chained CTEs, in later CTEs).
- **Readability** — instead of three levels of nesting, you write three named steps that read top-to-bottom like a small program.
- **Composability** — multiple CTEs in one `WITH` clause, separated by commas, build a pipeline.

Note the optimizer's relationship to CTEs has changed over time. Older PostgreSQL (pre-12) treated CTEs as an optimization fence and materialized them. Modern Postgres, DuckDB, and most other engines inline CTEs by default and let the planner reorder. Use `MATERIALIZED` or `NOT MATERIALIZED` hints if you need to override the default.

In [ ]:
%%sql
-- A two-step pipeline as a CTE chain. Each step has a name and reads cleanly
-- top-to-bottom: first aggregate, then label, then join.
WITH customer_totals AS (
    SELECT customer_id,
           COUNT(*)    AS order_count,
           SUM(amount) AS total
    FROM orders
    GROUP BY customer_id
),
labeled AS (
    SELECT customer_id,
           total,
           CASE WHEN total >= 100 THEN 'big spender' ELSE 'small spender' END AS tier
    FROM customer_totals
)
SELECT c.name, l.total, l.tier
FROM customers c
JOIN labeled l ON l.customer_id = c.customer_id
ORDER BY l.total DESC;

## 6. Recursive CTEs — hierarchies and graph traversal

Some questions are inherently recursive: "give me every direct and indirect report of Carol", "give me the full ancestor chain of this category", "give me all nodes reachable from this graph vertex". `WITH RECURSIVE` is how SQL expresses them.

A recursive CTE has two parts, glued together by `UNION ALL`:

1. **Anchor** — a non-recursive query that produces the starting rows.
2. **Recursive step** — a query that joins the CTE name back to itself, producing the next layer.

The engine evaluates the anchor once, then evaluates the recursive step repeatedly, feeding the new rows back in, until the recursive step produces zero rows. **Without a clear termination condition, the recursion never ends** — give the CTE a depth column you can cap if you don't fully trust the data.

In [ ]:
%%sql
-- Walk the org chart from the top down.
-- Anchor: people with no manager (the root of the tree).
-- Recursive: anyone whose manager is already in the result set.
WITH RECURSIVE chain AS (
    SELECT employee_id, name, manager_id, 0 AS depth
    FROM employees
    WHERE manager_id IS NULL
    UNION ALL
    SELECT e.employee_id, e.name, e.manager_id, c.depth + 1
    FROM employees e
    JOIN chain c ON e.manager_id = c.employee_id
)
SELECT depth, name
FROM chain
ORDER BY depth, name;

## 7. Common pitfalls (carry forward)

- **Scalar subquery returns more than one row** — runtime error. If you expect at most one but the data may produce more, add `LIMIT 1` with an explicit `ORDER BY`, or rewrite as a join.
- **`NOT IN` with a nullable subquery** — same trap as notebook 03. Replace with `NOT EXISTS`, which has clean `NULL` semantics, or filter `NULL`s out of the subquery explicitly.
- **Correlated subqueries can be expensive** — conceptually they run once per outer row. Modern planners often rewrite them into joins, but not always, and not on every engine. Check the `EXPLAIN` plan when performance matters.
- **Forgetting the alias on a derived table** — every subquery in `FROM` needs a name. The engine refuses unaliased ones.
- **Treating CTEs as performance-neutral** — they used to be optimization fences in older Postgres. Even today, an inlined CTE referenced N times can be evaluated N times unless materialized. For expensive CTEs used multiple times, `MATERIALIZED` (where supported) forces single evaluation.
- **Runaway recursive CTEs** — without a terminating condition, the recursion never stops. Always include a depth counter you can cap, and make sure your join condition cannot cycle (or use `UNION` instead of `UNION ALL` to dedupe rows in a graph traversal).
- **Column-count mismatch between anchor and recursive step** — both parts of a recursive CTE must produce the same number of columns with compatible types, in the same order.

Next up: **notebook 07 — Window Functions**, where we compute per-row aggregates without collapsing rows — running totals, ranks, lag/lead, and the `OVER` clause that drives all of them.